In [3]:
!uv pip list | grep ragas

Using Python 3.12.3 environment at: /home/vsc/LLM_TUNE/QA-FineTune/venv/venv-RAGAS
ragas                     0.2.14


In [2]:
! git clone https://huggingface.co/datasets/explodinggradients/Sample_non_english_corpus

fatal: 대상 경로가('Sample_non_english_corpus') 이미 있고 빈 디렉터리가 아닙니다.


In [3]:
! git clone https://huggingface.co/datasets/explodinggradients/Sample_Docs_Markdown

fatal: 대상 경로가('Sample_Docs_Markdown') 이미 있고 빈 디렉터리가 아닙니다.


In [1]:
from pathlib import Path
from langchain_core.documents import Document
import pandas as pd

data_path = Path("../data/data.xlsx")
df = pd.read_excel(data_path)

docs= []
for i, row in df.iterrows():
    metadata = {
        "faq_id": i,
        "filename": f"faq_row_{i}",  # 각 행에 고유 이름을 부여
        "source": "manual_excel"
    }
    content = f"제목: {row.get('TITLE', '')}\n내용: {row.get('DES', '')}"
    docs.append(Document(page_content=content, metadata=metadata))

In [2]:
len(docs)
print(docs[0])

page_content='제목: 회원증을 대리발급 할 수 있나요?
내용: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.

· 대상 : 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부
· 구비서류
  - 공통 : ①위임자 신분증, ②피위임자(대리인) 신분증
  - 장애인 : 장애인 복지카드 또는 장애인 증명서
  - 임신부 : 산모수첩 / 산모 : 주민등록등본(출산 후 12개월까지)
· 방법 : 홈페이지 회원가입 후 위 항목의 해당하는 구비서류를 지참하여 피위임자(대리인)이 도서관 방문' metadata={'faq_id': 0, 'filename': 'faq_row_0', 'source': 'manual_excel'}


In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

generator_llm = LangchainLLMWrapper(ChatOpenAI(
        model="/models/Exaone-3.5-32B-Instruct",
        openai_api_key="EMPTY",
        openai_api_base="http://localhost:8002/v1",
        max_tokens=2048,
        temperature=0.3,
    ))

keyphrase_extractor = KeyphraseExtractor(llm=generator_llm)

generator_embeddings = HuggingFaceEmbeddings(
        model_name="dragonkue/snowflake-arctic-embed-l-v2.0-ko",
        model_kwargs={'device': 'cpu'},
    )

docstore = InMemoryDocumentStore(
    splitter=splitter,
    embeddings=ragas_embeddings,
    extractor=keyphrase_extractor,
)

ModuleNotFoundError: No module named 'ragas.testset.extractor'

In [5]:
import nest_asyncio
import pandas as pd
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import apply_transforms
from ragas.testset.transforms import (
    KeyphrasesExtractor,
    OverlapScoreBuilder,
    SummaryExtractor
)

# 1. 비동기 환경 설정
nest_asyncio.apply()

# 2. 지식 그래프 재초기화 (깨끗한 상태에서 시작)
kg = KnowledgeGraph()
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={
                "page_content": doc.page_content,
                "document_metadata": doc.metadata,
            },
        )
    )

# 제목 + 내용에서 핵심 키워드를 뽑아 비슷한 주제끼리 연결
keyphrase_extractor = KeyphrasesExtractor( 
    llm=generator_llm,
    property_name="keyphrases",
    max_num=8,           # FAQ는 키워드 5~10개면 충분
)

# 키프레이즈가 곂친다면 관계를 만들어 줌
relation_builder = OverlapScoreBuilder(
    property_name="keyphrases",
    new_property_name="overlap_score",
    threshold=0.05,      # FAQ는 키워드 겹침이 적을 수 있으니 살짝 낮춰도 ok
    distance_threshold=0.85,
)

summary_extractor = SummaryExtractor(llm=generator_llm)

transforms = [
    summary_extractor,
    keyphrase_extractor,
    relation_builder,
]

apply_transforms(kg, transforms=transforms)

# 7. 즉각적인 결과 검증
print(f"\n✅ 지식 그래프 생성 완료!")
print(f"- 노드 수: {len(kg.nodes)}")
print(f"- 관계 수: {len(kg.relationships)}")

from pathlib import Path

kg_file = Path("my_faq_knowledge_graph.json")

# 저장
kg.save(kg_file)
print(f"\n✅ Knowledge Graph를 JSON으로 저장 완료: {kg_file}")

# (필요할 때) 불러오기 예시 - 다음 실행부터 이렇게 시작 가능
# loaded_kg = KnowledgeGraph.load(kg_file)
# print(f"불러온 KG - 노드 수: {len(loaded_kg.nodes)}, 관계 수: {len(loaded_kg.relationships)}")

Applying SummaryExtractor:   0%|          | 0/110 [00:00<?, ?it/s]

Applying KeyphrasesExtractor:   0%|          | 0/110 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]


✅ 지식 그래프 생성 완료!
- 노드 수: 110
- 관계 수: 473

✅ Knowledge Graph를 JSON으로 저장 완료: my_faq_knowledge_graph.json


In [6]:
if len(kg.nodes) > 0:
    sample_node = kg.nodes[0]
    sample_props = sample_node.properties
    
    print(f"\n🔍 첫 번째 노드 (Document) 샘플 속성:")
    print(f"  • page_content (앞 200자):")
    print(f"    {sample_props.get('page_content', '없음')[:200]}...")
    
    print(f"  • document_metadata:")
    print(f"    {sample_props.get('document_metadata', '없음')}")
    
    print(f"  • summary (SummaryExtractor 결과):")
    print(f"    {sample_props.get('summary', '아직 생성되지 않음')}")
    
    print(f"  • keyphrases (KeyphrasesExtractor 결과):")
    print(f"    {sample_props.get('keyphrases', '아직 생성되지 않음')}")


🔍 첫 번째 노드 (Document) 샘플 속성:
  • page_content (앞 200자):
    제목: 회원증을 대리발급 할 수 있나요?
내용: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.

· 대상 : 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부
· 구비서류
  - 공통 : ①위임자 신분증, ②피위임자(대리인) 신분증
  -...
  • document_metadata:
    {'faq_id': 0, 'filename': 'faq_row_0', 'source': 'manual_excel'}
  • summary (SummaryExtractor 결과):
    회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부를 대상으로 하며, 위임자와 피위임자의 신분증 외에 대상에 따라 추가 서류(장애인은 복지카드 또는 증명서, 임산부는 산모수첩 또는 출산 후 1년 이내의 주민등록등본)가 필요합니다. 대리인이 직접 도서관을 방문하여 신청할 수 있습니다.
  • keyphrases (KeyphrasesExtractor 결과):
    ['회원증 대리발급', '만14세 미만 아동', '만65세 이상 어르신', '장애인', '임산부', '위임자 신분증', '피위임자(대리인) 신분증', '장애인 복지카드']


In [7]:
if len(kg.relationships) > 0:
    print(f"\n  • 첫 번째 관계 예시:")
    rel = kg.relationships[0]
    
    # 핵심: .source.id / .target.id 사용
    source_id = rel.source.id
    target_id = rel.target.id
    
    print(f"    {source_id} → {target_id}")
    print(f"      (type: {rel.type}, score: {rel.properties.get('overlap_score', 'N/A')})")
    
    # 보너스: 관계된 노드의 내용 일부도 볼 수 있게
    print(f"      Source 문서 미리보기: {rel.source.properties.get('page_content', '')[:100]}...")
    print(f"      Target 문서 미리보기: {rel.target.properties.get('page_content', '')[:100]}...")


  • 첫 번째 관계 예시:
    d491500b-886c-4e29-9772-8be7b7b55b7c → 4c99303c-db17-4ef9-a6b1-977a2854b478
      (type: keyphrases_overlap, score: N/A)
      Source 문서 미리보기: 제목: 사립 작은도서관은 무엇인가요 ?
내용: 사립작은도서관은 다양한 주체가 다양한 방법으로 설립,운영하는 민간분야의 작은도서관 입니다._x000D_
고양시에는 현재 기준(2025...
      Target 문서 미리보기: 제목: 일산도서관 이용시간이 어떻게 되나요?
내용: ○ 종합자료실, 연속간행물실, 디지털자료실
(평일) 09:00 ~ 22:00 / (주말) 09:00 ~ 18:00
○ 어린이자료...


In [2]:
!pip install --upgrade ragas langchain langchain-openai langchain-huggingface

# 버전 확인 (실행 후 출력 확인)
import ragas
print("ragas version:", ragas.__version__)  # 0.4.0 이상이 나와야 함

import langchain
print("langchain version:", langchain.__version__)

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: langchain-huggingface
    Found existing installation: langchain-huggingface 1.2.0
    Uninstalling langchain-huggingface-1.2.0:
      Successfully uninstalled langchain-huggingface-1.2.0
ragas version: 0.4.3
langchain version: 1.2.10


In [1]:
import nest_asyncio
import asyncio
import pandas as pd
from pathlib import Path
from uuid import UUID

# ragas 관련
from ragas.testset.graph import KnowledgeGraph
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# synthesizer
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer

# LangChain
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

nest_asyncio.apply()

In [5]:
from ragas.llms import llm_factory
from openai import OpenAI  # OpenAI 호환 클라이언트

# Exaone 서버에 맞는 OpenAI 클라이언트 생성
exaone_client = OpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8002/v1",
)

# llm_factory로 Exaone 래핑
generator_llm = llm_factory(
    model="/models/Exaone-3.5-32B-Instruct",  # 모델 이름은 서버에서 인식하는 대로
    client=exaone_client,
    temperature=0.1,
    max_tokens=2048,
    # response_format={"type": "json_object"}  # 필요하면 여기 넣기 (서버 지원 여부 확인)
)

print("Exaone을 llm_factory로 설정 완료!")

from ragas.embeddings.base import embedding_factory
generator_embeddings = embedding_factory("huggingface", model="dragonkue/snowflake-arctic-embed-l-v2.0-ko")

Exaone을 llm_factory로 설정 완료!


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [6]:
kg_file = Path("my_faq_knowledge_graph.json")

try:
    loaded_kg = KnowledgeGraph.load(kg_file)
    print(f"KG 불러오기 성공!")
    print(f"노드 수: {len(loaded_kg.nodes)}")
    print(f"관계 수: {len(loaded_kg.relationships)}")
except Exception as e:
    print("KG 로드 실패:", e)
    # 실패하면 원본 JSON 경로/형식 다시 확인

KG 불러오기 성공!
노드 수: 110
관계 수: 473


In [7]:
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_kg,
    persona_list=[]   # 페르소나 자동 생성 OFF (빈 결과 방지)
)

# SingleHop만 사용 (MultiHop은 나중에)
synthesizer = SingleHopSpecificQuerySynthesizer(llm=generator_llm)

distribution = [
    (synthesizer, 1.0),
]

# 한국어 프롬프트 적용 (0.4.x에서 더 잘 동작)
async def adapt_prompts():
    prompts = await synthesizer.adapt_prompts("korean", llm=generator_llm)
    synthesizer.set_prompts(**prompts)
    print("한국어 프롬프트 적용 완료")

asyncio.run(adapt_prompts())

한국어 프롬프트 적용 완료


In [ ]:
# KG → LangChain Document로 변환
langchain_docs = [
    Document(
        page_content=node.properties.get("page_content", ""),
        metadata={
            **node.properties.get("document_metadata", {}),
            "summary": node.properties.get("summary", ""),
            "keyphrases": node.properties.get("keyphrases", [])
        }
    )
    for node in loaded_kg.nodes
]

print(f"변환된 Document 수: {len(langchain_docs)}")

# 생성 실행
try:
    testset = generator.generate_with_langchain_docs(
        documents=langchain_docs,
        testset_size=20,                    # 처음엔 10~20 추천
        query_distribution=distribution,         # 0.4.x에서는 distributions (s 복수)
        with_debugging_logs=True,
        raise_exceptions=True
    )
    
    print(f"생성된 샘플 수: {len(testset.samples)}")
    
    # 결과 DataFrame
    df = testset.to_pandas()
    display(df.head(10))  # Jupyter에서 예쁘게 보기
    df.to_csv("my_ragas_testset_0.4.csv", index=False)
    print("저장 완료: my_ragas_testset_0.4.csv")

except Exception as e:
    print("생성 실패:", str(e))
    print("로그를 자세히 확인하세요 (with_debugging_logs=True 덕분에 출력됨)")

변환된 Document 수: 110


Applying SummaryExtractor:   0%|          | 0/73 [00:00<?, ?it/s]